# ST-GNN Model Results

Visualizza i risultati del modello salvato da `python main.py`.

> **Pre-requisito**: eseguire `python main.py` dalla root del progetto.

In [ ]:
import sys, json
from pathlib import Path
sys.path.insert(0, str(Path('..').resolve()))

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import xarray as xr

from main import _normalize_dataset
from physiq_pv.model.st_gnn import STGNN
from physiq_pv.model.graph_builder import build_graph
from physiq_pv.data.dataset import PVDataset
from physiq_pv.data.quality_score import compute_qs
from torch.utils.data import DataLoader

sns.set_theme(style='darkgrid')
CKPT = '../checkpoints'

with open(f'{CKPT}/model_config.json') as f:
    cfg = json.load(f)
with open(f'{CKPT}/loss_history.json') as f:
    loss_history = json.load(f)

model = STGNN(**cfg)
model.load_state_dict(torch.load(f'{CKPT}/model.pt', map_location='cpu'))
model.eval()
print(f'Model: {sum(p.numel() for p in model.parameters()):,} params')
print(f'Loss history: {[round(l, 4) for l in loss_history]}')

ds = _normalize_dataset(xr.open_dataset('../data/real_data_dataset.nc'))
qs = compute_qs(ds)
edge_index, edge_weight = build_graph(ds['lat'].values, ds['lon'].values)

dataset = PVDataset(ds, qs)
loader  = DataLoader(dataset, batch_size=64, shuffle=False)
print(f'Dataset: {len(dataset)} samples')

In [ ]:
# Loss curve
fig, ax = plt.subplots(figsize=(8, 4))
epochs = range(1, len(loss_history) + 1)
ax.plot(epochs, loss_history, 'o-', color='steelblue', linewidth=2, markersize=6)
ax.set_xlabel('Epoch')
ax.set_ylabel('Physics Loss')
ax.set_title('ST-GNN Training Loss (Piedmont 2019)')
ax.grid(True, alpha=0.3)
for i, v in enumerate(loss_history):
    ax.annotate(f'{v:.4f}', (i + 1, v), textcoords='offset points',
                xytext=(0, 8), ha='center', fontsize=8)
plt.tight_layout()
plt.show()

print("=== LOSS CURVE ===")
for i, v in enumerate(loss_history):
    drop = f"  ({(loss_history[i-1]-v)/loss_history[i-1]*100:+.1f}%)" if i > 0 else ""
    print(f"  Epoch {i+1}: {v:.4f}{drop}")
total_drop = (loss_history[0] - loss_history[-1]) / loss_history[0] * 100
print(f"  Total drop: {total_drop:.1f}%")
converged = abs(loss_history[-1] - loss_history[-2]) / loss_history[-2] < 0.05
print(f"  Converged (last delta <5%): {converged}")

In [ ]:
# Inference: collect pred vs actual
all_pred_pv, all_true_pv = [], []
all_pred_ghi, all_true_ghi = [], []

with torch.no_grad():
    for i, (x, y_ghi, y_pv, qs_b, eta) in enumerate(loader):
        if i >= 200:
            break
        pg, pp = model(x, edge_index, edge_weight)
        all_pred_pv.append(pp.numpy())
        all_true_pv.append(y_pv.numpy())
        all_pred_ghi.append(pg.numpy())
        all_true_ghi.append(y_ghi.numpy())

pred_pv  = np.concatenate(all_pred_pv).ravel()
true_pv  = np.concatenate(all_true_pv).ravel()
pred_ghi = np.concatenate(all_pred_ghi).ravel()
true_ghi = np.concatenate(all_true_ghi).ravel()

day_mask   = true_ghi > 0.01
pred_pv_d, true_pv_d   = pred_pv[day_mask],  true_pv[day_mask]
pred_ghi_d, true_ghi_d = pred_ghi[day_mask], true_ghi[day_mask]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, (p, t, label) in zip(axes, [
    (pred_pv_d,  true_pv_d,  'PV Output (normalised)'),
    (pred_ghi_d, true_ghi_d, 'GHI [kW/m²]'),
]):
    lim = max(np.percentile(np.abs(t), 99), np.percentile(np.abs(p), 99))
    ax.scatter(t, p, s=2, alpha=0.2, color='steelblue')
    ax.plot([-lim, lim], [-lim, lim], 'r--', linewidth=1.5, label='ideal')
    corr = float(np.corrcoef(t, p)[0, 1])
    mae  = float(np.mean(np.abs(t - p)))
    ax.set_xlabel(f'Actual {label}')
    ax.set_ylabel(f'Predicted {label}')
    ax.set_title(f'{label}\nr={corr:.3f}  MAE={mae:.4f}')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

plt.suptitle('ST-GNN: Predicted vs Actual (daytime only)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

# --- Text report ---
def _metrics(p, t, name):
    corr  = float(np.corrcoef(t, p)[0, 1])
    mae   = float(np.mean(np.abs(t - p)))
    rmse  = float(np.sqrt(np.mean((t - p)**2)))
    bias  = float(np.mean(p - t))
    neg   = int((p < 0).sum())
    print(f"  {name}:")
    print(f"    r={corr:.3f}  MAE={mae:.4f}  RMSE={rmse:.4f}  bias={bias:+.4f}")
    print(f"    actual  range: [{t.min():.3f}, {t.max():.3f}]  mean={t.mean():.3f}")
    print(f"    predicted range: [{p.min():.3f}, {p.max():.3f}]  mean={p.mean():.3f}")
    print(f"    negative predictions: {neg} ({neg/len(p)*100:.1f}%)")

print(f"\n=== SCATTER METRICS (daytime, n={day_mask.sum():,}) ===")
_metrics(pred_pv_d,  true_pv_d,  "PV Output (normalised)")
_metrics(pred_ghi_d, true_ghi_d, "GHI [kW/m²]")

In [ ]:
# Time series: 3 sample plants
T_SHOW = 500
sample_plant_ids = [0, 47, 94]

seq_pred_pv  = {p: [] for p in sample_plant_ids}
seq_true_pv  = {p: [] for p in sample_plant_ids}
seq_true_ghi = {p: [] for p in sample_plant_ids}

with torch.no_grad():
    for i, (x, y_ghi, y_pv, qs_b, eta) in enumerate(loader):
        if i * loader.batch_size >= T_SHOW:
            break
        pg, pp = model(x, edge_index, edge_weight)
        for p in sample_plant_ids:
            seq_pred_pv[p].append(pp[:, p].numpy())
            seq_true_pv[p].append(y_pv[:, p].numpy())
            seq_true_ghi[p].append(y_ghi[:, p].numpy())

fig, axes = plt.subplots(len(sample_plant_ids), 1,
                         figsize=(14, 4 * len(sample_plant_ids)), sharex=True)
print("=== TIME SERIES METRICS ===")
for ax, p in zip(axes, sample_plant_ids):
    t_pv  = np.concatenate(seq_true_pv[p])[:T_SHOW]
    p_pv  = np.concatenate(seq_pred_pv[p])[:T_SHOW]
    t_ghi = np.concatenate(seq_true_ghi[p])[:T_SHOW]
    steps = np.arange(len(t_pv))

    corr = float(np.corrcoef(t_pv, p_pv)[0, 1]) if t_pv.std() > 0 else float('nan')
    mae  = float(np.mean(np.abs(t_pv - p_pv)))
    rmse = float(np.sqrt(np.mean((t_pv - p_pv)**2)))
    bias = float(np.mean(p_pv - t_pv))
    amp_ratio = float(p_pv.max() / t_pv.max()) if t_pv.max() > 0 else float('nan')

    ax.fill_between(steps, 0, t_ghi * 2, alpha=0.12, color='orange', label='GHI proxy')
    ax.plot(steps, t_pv, color='steelblue', linewidth=1.2, label='Actual PV')
    ax.plot(steps, p_pv, color='tomato', linewidth=1.0, linestyle='--', label='Predicted PV')
    ax.set_ylabel('PV output (norm.)')
    ax.set_title(f'Plant {p}  —  r={corr:.3f}  MAE={mae:.4f}  bias={bias:+.4f}  amp_ratio={amp_ratio:.2f}')
    ax.legend(loc='upper right', fontsize=8)
    ax.grid(True, alpha=0.3)

    print(f"  Plant {p:>2}: r={corr:.3f}  MAE={mae:.4f}  RMSE={rmse:.4f}  bias={bias:+.4f}")
    print(f"           actual  peak={t_pv.max():.3f}  mean={t_pv.mean():.3f}")
    print(f"           predicted peak={p_pv.max():.3f}  mean={p_pv.mean():.3f}  amp_ratio={amp_ratio:.2f}")

axes[-1].set_xlabel('Timestep (hours)')
plt.suptitle('ST-GNN: Time Series — 3 Sample Plants', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()